<a href="https://colab.research.google.com/github/MalakSalehh/Thesis-datasets/blob/main/ml2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install xgboost scikit-learn pandas numpy scipy

In [ ]:
import re
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split, KFold, cross_val_predict, RandomizedSearchCV
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import mean_absolute_error, mean_squared_error, cohen_kappa_score
from scipy.stats import pearsonr
from scipy.sparse import hstack, csr_matrix

from xgboost import XGBRegressor

In [ ]:
from google.colab import files
uploaded = files.upload()

Saving training_set_rel3.tsv.zip to training_set_rel3.tsv.zip


In [ ]:
!unzip training_set_rel3.tsv.zip

Archive:  training_set_rel3.tsv.zip
  inflating: training_set_rel3.tsv   


In [ ]:
df = pd.read_csv('/content/training_set_rel3.tsv', sep='\t', encoding='latin1')
df.head()

,essay_id,essay_set,essay,rater1_domain1,rater2_domain1,rater3_domain1,domain1_score,rater1_domain2,rater2_domain2,domain2_score,...,rater2_trait3,rater2_trait4,rater2_trait5,rater2_trait6,rater3_trait1,rater3_trait2,rater3_trait3,rater3_trait4,rater3_trait5,rater3_trait6
0,1,1,"Dear local newspaper, I think effects computer...",4,4,NaN,8,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2,1,"Dear @CAPS1 @CAPS2, I believe that using compu...",5,4,NaN,9,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,3,1,"Dear, @CAPS1 @CAPS2 @CAPS3 More and more peopl...",4,3,NaN,7,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,4,1,"Dear Local Newspaper, @CAPS1 I have found that...",5,5,NaN,10,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,5,1,"Dear @LOCATION1, I know having computers has a...",4,4,NaN,8,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [ ]:
set1 = df[df['essay_set'] == 1].copy()
set1.shape

(1783, 28)

# **CLEANING**

In [ ]:
def clean_text(text):
    text = str(text)
    text = text.replace('\n', ' ')
    text = re.sub(r'\s+', ' ', text).strip()
    return text

set1['essay_clean'] = set1['essay'].apply(clean_text)

Linguistic features

In [ ]:
def word_count(text):
    return len(text.split())

def sentence_count(text):
    sentences = re.split(r'[.!?]+', text)
    sentences = [s for s in sentences if s.strip()]
    return len(sentences)

def avg_word_len(text):
    words = text.split()
    if not words:
        return 0
    return np.mean([len(w) for w in words])

set1['word_count'] = set1['essay_clean'].apply(word_count)
set1['sentence_count'] = set1['essay_clean'].apply(sentence_count)
set1['avg_word_len'] = set1['essay_clean'].apply(avg_word_len)

define x and y


In [ ]:
X_text = set1['essay_clean']
y = set1['domain1_score'].astype(int)

. Create TF-IDF features

In [ ]:
tfidf = TfidfVectorizer(
    max_features=5000,
    ngram_range=(1, 2),
    stop_words='english'
)

X_tfidf = tfidf.fit_transform(X_text)

**Combine TF-IDF with linguistic features**

In [ ]:
extra_features = set1[['word_count', 'sentence_count', 'avg_word_len']].values
X_extra = csr_matrix(extra_features)

X = hstack([X_tfidf, X_extra])

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,

)


**Train a baseline XGBoost model**

In [ ]:
xgb_model = XGBRegressor(
    objective='reg:squarederror',
    n_estimators=200,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42
)

xgb_model.fit(X_train, y_train)

XGBRegressor(base_score=None, booster=None, callbacks=None,
             colsample_bylevel=None, colsample_bynode=None,
             colsample_bytree=0.8, device=None, early_stopping_rounds=None,
             enable_categorical=False, eval_metric=None, feature_types=None,
             feature_weights=None, gamma=None, grow_policy=None,
             importance_type=None, interaction_constraints=None,
             learning_rate=0.05, max_bin=None, max_cat_threshold=None,
             max_cat_to_onehot=None, max_delta_step=None, max_depth=6,
             max_leaves=None, min_child_weight=None, missing=nan,
             monotone_constraints=None, multi_strategy=None, n_estimators=200,
             n_jobs=None, num_parallel_tree=None, ...)

In [ ]:
y_pred = xgb_model.predict(X_test)
y_pred_round = np.rint(y_pred).astype(int)
y_pred_round = np.clip(y_pred_round, 2, 12)

In [ ]:
qwk = cohen_kappa_score(y_test, y_pred_round, weights='quadratic')
kappa = cohen_kappa_score(y_test, y_pred_round)
pcc, _ = pearsonr(y_test, y_pred_round)
mae = mean_absolute_error(y_test, y_pred_round)
rmse = np.sqrt(mean_squared_error(y_test, y_pred_round))

print("QWK:", qwk)
print("Cohen's Kappa:", kappa)
print("PCC:", pcc)
print("MAE:", mae)
print("RMSE:", rmse)

QWK: 0.8437572474881609
Cohen's Kappa: 0.37578632152259983
PCC: 0.8472825379387339
MAE: 0.5434173669467787
RMSE: 0.8096061912275311


In [ ]:
results_df = pd.DataFrame({
    'essay_id': set1.iloc[y_test.index]['essay_id'].values if hasattr(y_test, 'index') else np.nan,
    'human_score': y_test.values,
    'predicted_score': y_pred_round
})

results_df.head(20)

,essay_id,human_score,predicted_score
0,827,6,6
1,1477,6,7
2,234,8,9
3,801,9,9
4,780,9,10
5,271,8,8
6,419,6,7
7,1441,8,8
8,1391,8,8
9,112,9,9


In [ ]:
def score_category(score):
    if 2 <= score <= 5:
        return 'Low'
    elif 6 <= score <= 8:
        return 'Medium'
    else:
        return 'High'

results_df['human_category'] = results_df['human_score'].apply(score_category)
results_df['predicted_category'] = results_df['predicted_score'].apply(score_category)

results_df.head(20)

,essay_id,human_score,predicted_score,human_category,predicted_category
0,827,6,6,Medium,Medium
1,1477,6,7,Medium,Medium
2,234,8,9,Medium,High
3,801,9,9,High,High
4,780,9,10,High,High
5,271,8,8,Medium,Medium
6,419,6,7,Medium,Medium
7,1441,8,8,Medium,Medium
8,1391,8,8,Medium,Medium
9,112,9,9,High,High


In [ ]:
kf = KFold(n_splits=5, shuffle=True, random_state=42)

cv_preds = cross_val_predict(xgb_model, X, y, cv=kf)
cv_preds_round = np.rint(cv_preds).astype(int)
cv_preds_round = np.clip(cv_preds_round, 2, 12)

cv_qwk = cohen_kappa_score(y, cv_preds_round, weights='quadratic')
cv_kappa = cohen_kappa_score(y, cv_preds_round)
cv_pcc, _ = pearsonr(y, cv_preds_round)
cv_mae = mean_absolute_error(y, cv_preds_round)
cv_rmse = np.sqrt(mean_squared_error(y, cv_preds_round))

print("5-Fold CV QWK:", cv_qwk)
print("5-Fold CV Cohen's Kappa:", cv_kappa)
print("5-Fold CV PCC:", cv_pcc)
print("5-Fold CV MAE:", cv_mae)
print("5-Fold CV RMSE:", cv_rmse)

5-Fold CV QWK: 0.8336155387915295
5-Fold CV Cohen's Kappa: 0.3788748430156531
5-Fold CV PCC: 0.8407739538535964
5-Fold CV MAE: 0.5507571508693214
5-Fold CV RMSE: 0.8346129640591853
